In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import cv2
import sys
from PIL import Image
import torch
import torchvision.transforms.functional as TF

sys.path.append(str(Path.cwd().parent)) # Ajoute le dossier parent au chemin de recherche de Python

# Importation des fonctions utilitaires
from utils.stats_fct import overlay_mask

In [2]:
# Récupère l'excel avec les noms des echos et les catégories
metadata = pd.read_excel('../utils/correspondance_echo_final.xlsx')
names = metadata['new_name_file']
folder_path = Path(f'../U-NET/masks')

In [5]:
NUM_CLASSES = 3
COLS = 3

for name in names:
    # Vérifie la présence du nom dans le dossier masks
    files = list(folder_path.rglob(f'pred_{name}.txt'))

    display = []  # Liste pour stocker les images superposées
    model_names = []  # Liste pour stocker les noms des modèles

    if files:
        for file in files:
            model_names.append(str(file).split("/")[3])
            mask_path = file
            image_path = f'../data/COCO/images/{name}.jpg'

            # --- 1. Récupère l'image et applique le crop (enlève les bords noirs) ----
            x, y, w, h = 85, 33, 510, 380
            image = Image.open(image_path).convert("RGB")
            image = image.crop((x, y, x + w, y + h))

            # -- 2. Récupère le mask, directement à la taille du crop ----
            mask = np.zeros((NUM_CLASSES, h, w), dtype=np.float32)  # Crée un mask vide pour chaque classe

            with open(mask_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    class_id = int(parts[0])  # Récupère l'ID de la classe
                    coords = np.array(parts[1:]).reshape(-1, 2)

                    # Reconvertit les coordonnées relatives en pixels (relatifs au crop)
                    coords[:, 0] *= w
                    coords[:, 1] *= h

                    # Dessine la forme dans le canal correspondant à la classe
                    if 0 <= class_id < NUM_CLASSES:
                        cv2.fillPoly(mask[class_id], [coords.astype(np.int32)], 1.0)

            # Conversion en Tensor PyTorch
            mask_tensor = torch.from_numpy(mask)
            image_tensor = TF.to_tensor(image)

            # Superpose le mask sur l'image
            display.append(overlay_mask(image_tensor, mask_tensor, alpha=0.4, target_size=image.size))

        # ----- 4. Création de la figure combinée -----
        num_models = len(display) + 1  # Nombre total de modèles + 1 pour l'image originale
        rows = (num_models + COLS - 1) // COLS  # Calcul du nombre de lignes nécessaires
        fig, axes = plt.subplots(rows, COLS, figsize=(5 * COLS, 5 * rows))

        axes_flat = np.array(axes).reshape(-1) # Aplatit les axes pour faciliter l'itération

        # affiche l'image originale dans la première sous-figure
        axes_flat[0].imshow(image)
        axes_flat[0].set_title(f"{name}_original", fontsize=11, fontweight='bold', pad=8)
        axes_flat[0].axis('off')

        # Affichage des images dans les sous-figures
        for i, (img, m_name) in enumerate(zip(display, model_names)):
            ax = axes_flat[i+1]
            ax.imshow(img)
            ax.set_title(f"{name}_{m_name}", fontsize=11, fontweight='bold', pad=8)
            ax.axis('off')

        # Masque les cases de la grille inutilisées (si le total n'est pas un multiple de 3)
        for j in range(num_models, len(axes_flat)):
            axes_flat[j].axis('off')

        plt.tight_layout()

        # Sauvegarde
        results_dir = Path(f'visualisations/')
        results_dir.mkdir(parents=True, exist_ok=True)

        plt.savefig(results_dir / f'combined_{name}.png', bbox_inches='tight', dpi=300)
        plt.close(fig)

In [4]:
# Affiche une image présente dans "U-NET/results" avec les masques superposés
def display_image_with_masks(image_path, true_mask_path, pred_mask_path):
    # Charger l'image et les masques
    image = plt.imread(image_path)
    true_mask = plt.imread(true_mask_path)
    pred_mask = plt.imread(pred_mask_path)

    # Créer une figure avec 3 sous-graphes
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # Afficher l'image originale
    axs[0].imshow(image)
    axs[0].set_title('Image originale')
    axs[0].axis('off')

    # Afficher l'image avec le masque de vérité terrain superposé
    axs[1].imshow(image)
    axs[1].imshow(true_mask, alpha=0.5)  # Superposition avec transparence
    axs[1].set_title('Masque de vérité terrain')
    axs[1].axis('off')

    # Afficher l'image avec le masque prédit superposé
    axs[2].imshow(image)
    axs[2].imshow(pred_mask, alpha=0.5)  # Superposition avec transparence
    axs[2].set_title('Masque prédit')
    axs[2].axis('off')

    plt.tight_layout()
    plt.show()